# Hybrid and rerank

## Get started

<img src="hybrid_and_rerank.png">

## Prepare the data

我们使用 Langchain WebBaseLoader 从博客源加载文档，并通过 RecursiveCharacterTextSplitter 将其拆分为多个片段。

In [1]:
import os

CUSTOM_CACHE = r'F:\Teewon\Milvue\models'
os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(CUSTOM_CACHE, 'transformers')
os.environ['TORCH_HOME'] = CUSTOM_CACHE

In [2]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a WebBaseLoader instance to load documents from web sources
loader = WebBaseLoader(
    web_path=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content","post-title","post-header")
        ),# 只解析 HTML 中符合特定条件的部分
    )
)

# Load documents from web sources using the loader
documents=loader.load()

# Initialize a RecursiveCharacterTextSplitter for splitting text into chunks
text_spliiter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=0)

# Split the documents into chunks using the text_splitter
docs=text_spliiter.split_documents(documents)

# Inspect
docs[1]

C:\Users\Administrator\AppData\Local\Temp\ipykernel_12052\2095123501.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Short-term memory: I would consider all the in-context learning (See Prompt Engineering) as utilizing short-term memory of the model to learn.\nLong-term memory: This provides the agent with the capability to retain and recall (infinite) information over extended periods, often by leveraging an external vector store and fast retrieval.\n\n\nTool use\n\nThe agent learns to call external APIs for extra information that is missing from the model weights (often hard to change after pre-training), including current information, code execution capability, access to proprietary information sources and more.\n\n\n\n\n\nOverview of a LLM-powered autonomous agent system.')

## Build the chain

We load the docs into milvus vectorstore, and build a milvus retriever.

In [3]:
from rag_utils.vanilla import vectorstore,format_docs,rag_prompt,llm

vectorstore.add_documents(docs)
milvus_retriever = vectorstore.as_retriever()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

And build a bm25 retriever from the docs.

In [4]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever=BM25Retriever.from_documents(docs)

Build a vanilla RAG chain.

In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import PromptTemplate

vanilla_rag_chain = (
    {"context": milvus_retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

Prepare hybrid_and_rerank_retriever.

In [9]:
from rag_utils.hybrid_and_rerank import RerankerRunnable,CrossEncoderReranker

local_reranker=CrossEncoderReranker(
    model_name="cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_k=4
)


reranker=RerankerRunnable(
    compressor=local_reranker,
    top_k=4
)

hybrid_and_rerank_retriever={
    "milvus_retrieved_doc":milvus_retriever,
    "bm25_retrieved_doc":bm25_retriever,
    "query":RunnablePassthrough(),
}|reranker

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Build hybrid_and_rerank_chain.

In [10]:
hybrid_and_rerank_chain=(
    {"context":hybrid_and_rerank_retriever|format_docs,"question":RunnablePassthrough()}
    |rag_prompt
    | llm
    | StrOutputParser()
)

## Test the chain

In [11]:
query = "Which model use tools?"

vanilla_result=vanilla_rag_chain.invoke(query)
hybrid_and_rerank_result=hybrid_and_rerank_chain.invoke(query)
print(f"\n[vanilla_result]:\n{vanilla_result}\n\n[hybrid_and_rerank_result]:\n{hybrid_and_rerank_result}")

len(milvus_retrieved_doc)=4
len(bm25_retrieved_doc)=4
len(unique_documents)=6

[vanilla_result]:
The context indicates several models/frameworks that use tools:

- **TALM** (Tool Augmented Language Models) and **Toolformer**, which are fine-tuned to call external tool APIs.
- **ChatGPT** with Plugins and OpenAI API function calling.
- **HuggingGPT**, which uses ChatGPT as a task planner to select and use HuggingFace models.
- **ChemCrow**, a domain-specific agent augmented with **13 expert-designed tools** for chemistry tasks.

[hybrid_and_rerank_result]:
Based on the provided context, the models/frameworks that use tools include:

- **TALM** (Tool Augmented Language Models; Parisi et al. 2022)
- **Toolformer** (Schick et al. 2023)
- **ChatGPT Plugins and OpenAI API function calling**
- **HuggingGPT** (Shen et al. 2023), which uses ChatGPT as a task planner to select HuggingFace models
- **ChemCrow** (Bran et al. 2023), an LLM augmented with 13 expert-designed tools for chemistry/scien

| 方法 | 回答内容 | 评价 |
|---|---|---|
| **Vanilla RAG** | 列出了 TALM、Toolformer、ChatGPT with Plugins、HuggingGPT、ChemCrow，但未提供论文年份或作者，描述相对简略。 | ✅ 覆盖了主要工具使用框架，但缺少引用细节，学术性和信息完整度稍显不足。 |
| **Hybrid + Rerank** | 同样列出五个框架，但补充了论文发表年份和作者（如 Parisi et al. 2022、Schick et al. 2023、Shen et al. 2023、Bran et al. 2023），并对 ChemCrow 的工具数量（13个）和 HuggingGPT 的任务规划器角色进行了更详细的描述。 | ✅ 信息更丰富、精确，引入了文献引用，增强了可信度，描述更专业、详尽，整体质量更高。 |

### 📊 差异分析
- **覆盖完整性**：两者均列出了所有相关框架，覆盖度一致。
- **信息深度**：Hybrid + Rerank 通过融合语义检索（Milvus）和关键词检索（BM25），并去除重复、重排，筛选出了包含更多元数据（如论文引用）的文档块，因此能生成更详实、有据可查的回答。
- **可读性与可信度**：Hybrid 结果中明确的参考文献标注使回答更具专业性和说服力。

**结论**：混合检索（向量+BM25）配合重排，通过多路召回和去重重排，有效提升了检索结果的多样性和信息密度，从而显著提高了最终生成的回答质量。该策略特别适合需要精确引用或深度信息的专业问答场景。